In [ ]:
import random
GOAL = [[1,2,3],[4,5,6],[7,8,0]]
model = {
    "trang_thai_hien_tai": None,  
    "da_tham":             set(), 
    "lich_su":             [],   
}
def hien_thi(state):
    print("+---+---+---+")
    for hang in state:
        print("|" + " | ".join(" " if v == 0 else str(v) for v in hang) + "|")
        print("+---+---+---+")
    print(f"Da tham: {len(model['da_tham'])} trang thai  |  Lich su: {model['lich_su'][-5:]}\n")

def to_tuple(state):
    return tuple(tuple(h) for h in state)

def update_model(state, huong):
    model["trang_thai_hien_tai"] = state
    model["da_tham"].add(to_tuple(state))
    if huong:
        model["lich_su"].append(huong)
def sensor(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return (i, j)
def processor(state, blank):
    r, c = blank
    ung_vien = []
    if r > 0:
        moi = [h[:] for h in state]
        moi[r][c], moi[r-1][c] = moi[r-1][c], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("LEN", moi))
    if r < 2:
        moi = [h[:] for h in state]
        moi[r][c], moi[r+1][c] = moi[r+1][c], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("XUONG", moi))
    if c > 0:
        moi = [h[:] for h in state]
        moi[r][c], moi[r][c-1] = moi[r][c-1], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("TRAI", moi))
    if c < 2:
        moi = [h[:] for h in state]
        moi[r][c], moi[r][c+1] = moi[r][c+1], moi[r][c]
        if to_tuple(moi) not in model["da_tham"]:
            ung_vien.append(("PHAI", moi))

    if not ung_vien:
        return None, None 

    return random.choice(ung_vien)
def actuator(huong, state):
    print(f"-> Di chuyen blank: {huong}")
    return state
def main():
    print("=== 8-Puzzle Model-Based Agent ===\n")
    print("Nhap ma tran 3x3 (dung 0 cho o trong):")

    state = []
    for i in range(3):
        while True:
            raw = input(f"Hang {i+1}: ").split()
            if len(raw) == 3:
                state.append(list(map(int, raw)))
                break
            print("Nhap lai, can dung 3 so")

    print("\nTrang thai ban dau:")
    hien_thi(state)
    update_model(state, None)

    buoc = 0
    while state != GOAL:
        buoc += 1
        print(f"--- Buoc {buoc} ---")

        blank        = sensor(state)
        huong, state = processor(state, blank)

        if huong is None:
            print("Bi ket, khong con trang thai moi!")
            break

        state = actuator(huong, state)
        update_model(state, huong)
        hien_thi(state)

        if buoc > 5000:
            print("Qua nhieu buoc, dung lai.")
            break

    if state == GOAL:
        print(f"=== Da giai xong sau {buoc} buoc! ===")
        print(f"Toan bo lich su ({len(model['lich_su'])} buoc): {model['lich_su']}")

if __name__ == "__main__":
    main()

In [ ]:

model = {
    "trang_thai_hien_tai": None,  
    "da_tham":             set(), 
    "lich_su":             [],    
    "o_ban_con_lai":       [],   
}

def hien_thi(phong, pos):
    r, c = pos
    for i in range(3):
        hang = "|"
        for j in range(3):
            if (i, j) == (r, c):
                hang += " R |"
            elif phong[i][j] == 0:
                hang += "   |"
            elif phong[i][j] == 1:
                hang += " * |"
        print(hang)
    print(f"Da tham: {sorted(model['da_tham'])}  |  Con ban: {model['o_ban_con_lai']}\n")

def update_model(phong, pos, hanh_dong):
    model["trang_thai_hien_tai"] = (phong, pos)
    model["da_tham"].add(pos)
    if hanh_dong:
        model["lich_su"].append(hanh_dong)
    model["o_ban_con_lai"] = [
        (i, j)
        for i in range(3)
        for j in range(3)
        if phong[i][j] == 1
    ]

def sensor(phong, pos):
    r, c = pos
    return phong[r][c]

def processor(tinh_trang, pos, phong):
    r, c = pos
    if tinh_trang == 1:
        return "HUT"
    if not model["o_ban_con_lai"]:
        return "DUNG"
    dich = None
    kc_min = float("inf")
    for (i, j) in model["o_ban_con_lai"]:
        kc = abs(i - r) + abs(j - c)
        if kc < kc_min:
            kc_min = kc
            dich = (i, j)
    dr, dc = dich[0] - r, dich[1] - c

    if abs(dr) >= abs(dc):
        if dr < 0 and r - 1 >= 0 and (r-1, c) not in model["da_tham"]:
            return "LEN"
        if dr > 0 and r + 1 <= 2 and (r+1, c) not in model["da_tham"]:
            return "XUONG"
        if dc < 0 and c - 1 >= 0 and (r, c-1) not in model["da_tham"]:
            return "TRAI"
        if dc > 0 and c + 1 <= 2 and (r, c+1) not in model["da_tham"]:
            return "PHAI"
    else:
        if dc < 0 and c - 1 >= 0 and (r, c-1) not in model["da_tham"]:
            return "TRAI"
        if dc > 0 and c + 1 <= 2 and (r, c+1) not in model["da_tham"]:
            return "PHAI"
        if dr < 0 and r - 1 >= 0 and (r-1, c) not in model["da_tham"]:
            return "LEN"
        if dr > 0 and r + 1 <= 2 and (r+1, c) not in model["da_tham"]:
            return "XUONG"
    if r > 0: return "LEN"
    if r < 2: return "XUONG"
    if c > 0: return "TRAI"
    if c < 2: return "PHAI"

    return "DUNG"

def actuator(hanh_dong, pos, phong):
    r, c = pos

    if hanh_dong == "HUT":
        phong[r][c] = 0
        print(f"-> HUT bui tai ({r},{c})")
    elif hanh_dong == "LEN":
        pos = (r - 1, c)
        print(f"-> Di LEN den ({r-1},{c})")
    elif hanh_dong == "XUONG":
        pos = (r + 1, c)
        print(f"-> Di XUONG den ({r+1},{c})")
    elif hanh_dong == "TRAI":
        pos = (r, c - 1)
        print(f"-> Di TRAI den ({r},{c-1})")
    elif hanh_dong == "PHAI":
        pos = (r, c + 1)
        print(f"-> Di PHAI den ({r},{c+1})")
    elif hanh_dong == "DUNG":
        print(f"-> DUNG tai ({r},{c})")
    return pos, phong
def main():
    print("Nhap phong 3x3 (0=sach, 1=ban):")
    phong = []
    for i in range(3):
        while True:
            raw = input(f"Hang {i+1}: ").split()
            if len(raw) == 3 and all(x in "01" for x in raw):
                phong.append(list(map(int, raw)))
                break
            print("Nhap lai, chi dung 0 hoac 1")

    while True:
        raw = input("Vi tri robot (hang col, vi du: 0 0): ").split()
        if len(raw) == 2:
            r, c = int(raw[0]), int(raw[1])
            if 0 <= r <= 2 and 0 <= c <= 2:
                pos = (r, c)
                break
        print("Vi tri khong hop le")

    print()
    update_model(phong, pos, None)
    hien_thi(phong, pos)

    buoc = 0
    while model["o_ban_con_lai"]:
        buoc += 1
        tinh_trang = sensor(phong, pos)
        hanh_dong  = processor(tinh_trang, pos, phong)
        if hanh_dong == "DUNG":
            print("-> DUNG, khong the tiep tuc.")
            break
        pos, phong = actuator(hanh_dong, pos, phong)
        update_model(phong, pos, hanh_dong)
        hien_thi(phong, pos)

    if not model["o_ban_con_lai"]:
        print(f"Phong da sach sau {buoc} buoc")
        print(f"Lich su: {model['lich_su']}")
if __name__ == "__main__":
    main()